# PCA - Tarefa 01: *HAR* com PCA

Vamos trabalhar com a base da demonstração feita em aula, mas vamos explorar um pouco melhor como é o desempenho da árvore variando o número de componentes principais.

In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV
from time import time


base_dir = "C:/Users/gusta/Documents/EBAC/Módulo 27/Dados/UCI HAR Dataset"

filename_features = base_dir + "/features.txt"
filename_labels = base_dir + "/activity_labels.txt"

filename_subtrain = base_dir + "/train/subject_train.txt"
filename_xtrain = base_dir + "/train/X_train.txt"
filename_ytrain = base_dir + "/train/y_train.txt"

filename_subtest = base_dir + "/test/subject_test.txt"
filename_xtest = base_dir + "/test/X_test.txt"
filename_ytest = base_dir + "/test/y_test.txt"


features = pd.read_csv(filename_features, header=None, names=['nome_var'], sep="#")['nome_var']
labels = pd.read_csv(filename_labels, delim_whitespace=True, header=None, names=['cod_label', 'label'])

subject_train = pd.read_csv(filename_subtrain, header=None, names=['subject_id'])['subject_id']
X_train = pd.read_csv(filename_xtrain, delim_whitespace=True, header=None, names=features.tolist())
y_train = pd.read_csv(filename_ytrain, header=None, names=['cod_label'])

subject_test = pd.read_csv(filename_subtest, header=None, names=['subject_id'])['subject_id']
X_test = pd.read_csv(filename_xtest, delim_whitespace=True, header=None, names=features.tolist())
y_test = pd.read_csv(filename_ytest, header=None, names=['cod_label'])


## Árvore de decisão

Rode uma árvore de decisão com todas as variáveis, utilizando o ```ccp_alpha=0.001```. Avalie a acurácia nas bases de treinamento e teste. Avalie o tempo de processamento.

In [4]:
%%time

clf = DecisionTreeClassifier(random_state=1234, ccp_alpha = 0.001).fit(X_train, y_train)

train_score = clf.score(X_train, y_train)
test_score = clf.score(X_test, y_test)

print(f'Acurácia da melhor árvore na base de treino:    {train_score*100:.1f}')
print(f'Acurácia da melhor árvore na base de teste:     {test_score*100:.1f}')

Acurácia da melhor árvore na base de treino:    97.6
Acurácia da melhor árvore na base de teste:     88.0
CPU times: total: 5.28 s
Wall time: 5.56 s


## Árvore com PCA

Faça uma análise de componemtes principais das variáveis originais. Utilize apenas uma componente. Faça uma árvore de decisão com esta componente como variável explicativa.

- Avalie a acurácia nas bases de treinamento e teste
- Avalie o tempo de processamento

In [6]:
%%time

prcomp = PCA(n_components=1).fit(X_train)

pc_treino = prcomp.transform(X_train)
pc_teste  = prcomp.transform(X_test)

n = 1

colunas = ['cp'+str(x+1) for x in list(range(n))]

pc_train = pd.DataFrame(pc_treino[:,:n], columns = colunas)
pc_test  = pd.DataFrame( pc_teste[:,:n], columns = colunas)

clf = DecisionTreeClassifier(random_state=1234, ccp_alpha=0.001)
clf = clf.fit(pc_train, y_train)
train_score = clf.score(pc_train, y_train)
test_score = clf.score(pc_test,y_test)

print(f'Acurácia da melhor árvore na base de treino:    {train_score*100:.1f}')
print(f'Acurácia da melhor árvore na base de teste:     {test_score*100:.1f}')

Acurácia da melhor árvore na base de treino:    50.0
Acurácia da melhor árvore na base de teste:     45.7
CPU times: total: 406 ms
Wall time: 276 ms


A acurácia foi relativamente baixa em comparação com a árvore de decisão original, o que se justifica pelo uso de apenas uma variável para a análise. Por outro lado, o tempo de processamento foi significativamente menor, já que a árvore de decisão original levou cerca de 4,2 segundos, enquanto a árvore com PCA não consumiu nem 1 segundo.

## Testando o número de componentes

Com base no código acima, teste a árvore de classificação com pelo menos as seguintes possibilidades de quantidades de componentes: ```[1, 2, 5, 10, 50]```. Avalie para cada uma delas:

- Acurácia nas bases de treino e teste
- Tempo de processamento


In [9]:


componentes = [1, 2, 5, 10, 50]

train_accuracies = []
test_accuracies = []
tempos = []

for n in componentes:
    prcomp = PCA(n_components=n).fit(X_train)
    pc_treino = prcomp.transform(X_train)
    pc_teste  = prcomp.transform(X_test)
    
    clf = DecisionTreeClassifier(random_state=1234, ccp_alpha=0.001)
    
    # Medindo o tempo de treinamento
    start_train = time()
    clf.fit(pc_treino, y_train)
    end_train = time()
    treino_tempo = end_train - start_train
    
    train_pred = clf.predict(pc_treino)
    train_acc = accuracy_score(y_train, train_pred)
    
    # Medindo o tempo de teste
    start_test = time()
    test_pred = clf.predict(pc_teste)
    end_test = time()
    teste_tempo = end_train - start_train
    test_acc = accuracy_score(y_test, test_pred)
    
    # Somando tempo de treinamento e teste
    tempo_total = treino_tempo + teste_tempo
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    tempos.append(tempo_total)
    
    print(f"Número de componentes: {n}")
    print(f"Acurácia no treino: {train_acc*100:.1f}")
    print(f"Acurácia no teste: {test_acc*100:.1f}")
    print(f"Tempo: {tempo_total:.2f}s")
    print('\n')

Número de componentes: 1
Acurácia no treino: 50.0
Acurácia no teste: 45.7
Tempo: 0.07s


Número de componentes: 2
Acurácia no treino: 61.3
Acurácia no teste: 58.5
Tempo: 0.09s


Número de componentes: 5
Acurácia no treino: 84.6
Acurácia no teste: 78.9
Tempo: 0.13s


Número de componentes: 10
Acurácia no treino: 89.3
Acurácia no teste: 82.4
Tempo: 0.22s


Número de componentes: 50
Acurácia no treino: 91.9
Acurácia no teste: 82.3
Tempo: 1.38s




## Conclua

- O que aconteceu com a acurácia?
- O que aconteceu com o tempo de processamento?

Podemos observar que, conforme aumentamos o número de componentes (variáveis) analisadas na árvore utilizando PCA, tanto o tempo de processamento quanto a acurácia aumentam. Na última configuração, com 50 variáveis, a acurácia foi de aproximadamente 91,9% na base de treino e 82,3% na base de teste, em comparação com a árvore sem PCA, que obteve 97,6% na base de treino e 88,0% na base de teste.

No entanto, é importante destacar que, mesmo utilizando 50 variáveis, o tempo de processamento foi de apenas 1,06 segundos, significativamente inferior aos 4,2 segundos necessários para o modelo sem a aplicação do PCA.